In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/tmp/pocket-r1'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))

if os.path.exists(REPO_DIR):
    subprocess.check_call(['rm', '-rf', REPO_DIR])
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'bitsandbytes', 'accelerate', 'vllm'])
print('Phase 2 environment ready!')


In [ ]:
import os, shutil, subprocess, sys

SFT_DATA_DIR = '/tmp/sft_data'
OUT_REPORT_DIR = '/kaggle/working/exp1b_sft_fim'
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

# Build 8,000 verified standard CoT + 25% Masked Step Infilling (~9,500 train + ~400 val)
subprocess.check_call([
    sys.executable, f'{REPO_DIR}/scripts/build_sft_dataset.py',
    '--env', 'hu_nlhe',
    '--target-examples', '8000',
    '--base-seed', '700000',
    '--fim-ratio', '0.25',
    '--equity-samples', '48',
    '--output-dir', SFT_DATA_DIR,
    '--eval-suite', f'{REPO_DIR}/data/eval_suites/hu_nlhe_heldout_v1.jsonl',
], cwd=REPO_DIR)

shutil.copy(f'{SFT_DATA_DIR}/stats_report.json', f'{OUT_REPORT_DIR}/sft_stats_report.json')
print('SFT dataset built and stats saved!')


In [ ]:
import os, subprocess, sys

LORA_DIR = '/kaggle/working/exp1b_sft_fim_lora'
MERGED_DIR = '/tmp/exp1b_sft_fim_merged'

train_env = os.environ.copy()
train_env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.check_call([
    sys.executable, f'{REPO_DIR}/scripts/train_sft.py',
    '--config', f'{REPO_DIR}/configs/training/sft_nlhe_fim.yaml',
    '--train-file', f'{SFT_DATA_DIR}/train_mixed_fim.jsonl',
    '--val-file', f'{SFT_DATA_DIR}/val_sft.jsonl',
    '--output-dir', LORA_DIR,
    '--merged-dir', MERGED_DIR,
    '--epochs', '2',
    '--merge',
], cwd=REPO_DIR, env=train_env)
print('Training & LoRA merge complete!')


In [ ]:
import subprocess, time, urllib.request, sys, os

LOG_PATH = '/tmp/vllm_exp1b.log'
log_file = open(LOG_PATH, 'w')

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MERGED_DIR,
    '--served-model-name', 'Qwen/Qwen3-4B-exp1b-sft-fim',
    '--dtype', 'half',
    '--tensor-parallel-size', '2',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--host', '127.0.0.1',
    '--port', '8000',
]
print('Starting vLLM for Exp 1b:', ' '.join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

ready = False
for sec in range(420):
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH) as f:
            print('vLLM exited early! Last logs:\n', f.read()[-4000:])
        raise RuntimeError(f'vLLM crashed with code {vllm_proc.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {sec}s:', r.read().decode())
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    vllm_proc.terminate()
    raise TimeoutError('vLLM did not become ready within 420s')

try:
    eval_cmd = [
        sys.executable, f'{REPO_DIR}/scripts/run_exp0_baseline.py',
        '--config', f'{REPO_DIR}/configs/eval/exp0_baseline.yaml',
        '--backend', 'openai_compatible',
        '--model', 'Qwen/Qwen3-4B-exp1b-sft-fim',
        '--base-url', 'http://127.0.0.1:8000/v1',
        '--workers', '24',
        '--no-ev',
        '--experiment-name', 'exp1b_sft_fim',
        '--output-dir', OUT_REPORT_DIR,
    ]
    print('Evaluating Exp 1b on 2000 Held-out hands:', ' '.join(eval_cmd))
    subprocess.check_call(eval_cmd, cwd=REPO_DIR)
finally:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=20)
    except Exception:
        vllm_proc.kill()
    log_file.close()
    print('Phase 2 complete! Outputs:', os.listdir(OUT_REPORT_DIR))
